# Ampliador honesto de fotos — ejecución en Google Colab

Este cuaderno corre el proyecto en una GPU gratuita de Colab (T4). Sirve para quien no tiene placa de video.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU** → Guardar*.

Pasos:
1. Verificar la GPU.
2. Clonar el repositorio e instalar el entorno con `uv` (Python 3.12, versiones exactas de `uv.lock`).
3. *(Opcional)* Generar las evidencias de qué inventa el modelo.
4. Levantar la app con `uv run app.py` y abrir el **link público** que aparece.

## 1. Verificar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clonar el repositorio e instalar el entorno

`uv sync` crea el entorno con Python 3.12 y las versiones exactas del repo. Tarda unos minutos (baja PyTorch).

In [ ]:
REPO = "https://github.com/hbarrabasqui/tpi2-ampliador.git"  # cambialo si tu repo tiene otro nombre

!rm -rf /content/tpi2-ampliador
!git clone -q {REPO} /content/tpi2-ampliador
%cd /content/tpi2-ampliador
!pip install -q uv
!uv sync
!uv run python -c "import torch; print('PyTorch', torch.__version__, '| GPU visible:', torch.cuda.is_available())"

Si la última línea dice `GPU visible: False`, revisá el paso de la T4 y volvé a ejecutar.

Verificación rápida (no descarga modelos):

In [ ]:
!uv run python tests/verificar.py

## 3. (Opcional) Generar las evidencias de qué es real y qué es inventado

Amplía las 4 fotos de ejemplo y las compara con el **original grande** (la verdad). También prueba 3 semillas y 4 niveles de libertad. Tarda unos 5 a 10 minutos, más la descarga de los modelos (~3 GB) la primera vez.

In [ ]:
!PYTHONUNBUFFERED=1 uv run python scripts/generar_evidencias.py

In [ ]:
from IPython.display import Image, Markdown, display
import glob

display(Markdown(open("resultados/resumen.md", encoding="utf-8").read()))
for ruta in sorted(glob.glob("resultados/*_comparacion.png")) + sorted(glob.glob("resultados/*_semillas.png")):
    display(Markdown(f"**{ruta}**"))
    display(Image(ruta))

Guardar los resultados en tu Google Drive (para el README y el video):

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DESTINO = "/content/drive/MyDrive/IFTS 24 - Tecnicas de Procesamiento Digital de Imagenes (Clases)/TPI 2/resultados"
!mkdir -p "{DESTINO}"
!cp resultados/* "{DESTINO}/"
print("Copiado a", DESTINO)

## 4. Levantar la app

Ejecutá la celda y esperá la línea **`Running on public URL: https://....gradio.live`**. Abrí ese link en otra pestaña.

La celda queda "corriendo" mientras la app está abierta. Para cerrarla, tocá el botón de detener.

In [ ]:
!PYTHONUNBUFFERED=1 uv run app.py